In [1]:
import os
import time
import json
import shutil
import argparse
import numpy as np

import functools
import operator

import pickle, gzip

from topcoffea.modules.utils import regex_match,clean_dir,dict_comp
from ttbarEFT.modules.datacard_tools import *

In [2]:
def combine_vineReduce_channels(pkl): 
    vineReduce_dict = pickle.load(gzip.open(pkl_file))
    first_ch = next(iter(vineReduce_dict))
    variables = vineReduce_dict[first_ch].keys()
    
    hists = {}
    for var in variables: 
        h_list = [vineReduce_dict[ch][var] for ch in vineReduce_dict]
        hists[var] = functools.reduce(operator.add, h_list)

    return hists

def flatten_vineReduce_hists(pkl, outname = 'flat_hists'):
    # Flattens the histogram structure from vineReduce and saves as a new pickle. 
    # Returns the name of the new pickle file 
    
    flattend_hists = combine_vineReduce_channels(pkl)
    tmp_pkl = f"{outname}.pkl.gz"
    with gzip.open(tmp_pkl, "wb") as f:
        pickle.dump(flattend_hists, f)
    
    return tmp_pkl

In [4]:
pkl_file = "../analysis/SR_2017_2j2b.pkl.gz"
var = 'mllbb'
ch = 'ee_2j_2b'
out_dir = './test_cards/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

prepared_pkl = flatten_vineReduce_hists(pkl_file)

obj = DatacardMaker(
    prepared_pkl,
    year_lst=[],         # Processes all years found in pkl
    do_nuisance=True,    # Set to False if you want to skip systematics entirely for now
    out_dir=out_dir
)

my_wcs = ["ctGRe", "ctu8"] 
wcs_dict = {p: my_wcs for p in obj.processes(var)}
# print(wcs_dict)

print(f"Generating card for channel: {ch}")
obj.analyze(
    km_dist=var,
    ch=ch,
    selected_wcs=wcs_dict,
    crop_negative_bins=True,
    wcs_dict=wcs_dict
)



Opening: /users/hnelson2/ttbarEFT-coffea2025/ttbarEFT/params/rate_systs.json
Opening: flat_hists.pkl.gz
Pkl Open Time: 1.05 s
Loading: njets
Skipping (ignored): dataUL17
Correlating years
('ttUL17', 'ee_2j_2b', 'btagSFbc_2017Up') -- 
('ttUL17', 'ee_2j_2b', 'btagSFbc_2017Down') -- 
('ttUL17', 'ee_2j_2b', 'btagSFlight_2017Up') -- 
('ttUL17', 'ee_2j_2b', 'btagSFlight_2017Down') -- 
('ttUL17', 'mm_2j_2b', 'btagSFbc_2017Up') -- 
('ttUL17', 'mm_2j_2b', 'btagSFbc_2017Down') -- 
('ttUL17', 'mm_2j_2b', 'btagSFlight_2017Up') -- 
('ttUL17', 'mm_2j_2b', 'btagSFlight_2017Down') -- 
('ttUL17', 'em_2j_2b', 'btagSFbc_2017Up') -- 
('ttUL17', 'em_2j_2b', 'btagSFbc_2017Down') -- 
('ttUL17', 'em_2j_2b', 'btagSFlight_2017Up') -- 
('ttUL17', 'em_2j_2b', 'btagSFlight_2017Down') -- 
('tWUL17', 'ee_2j_2b', 'btagSFbc_2017Up') -- 
('tWUL17', 'ee_2j_2b', 'btagSFbc_2017Down') -- 
('tWUL17', 'ee_2j_2b', 'btagSFlight_2017Up') -- 
('tWUL17', 'ee_2j_2b', 'btagSFlight_2017Down') -- 
('tWUL17', 'mm_2j_2b', 'btagSFbc_201

In [ ]:
import uproot
f = uproot.open("test_cards/TESTING-ttbar-ee_2j_2b_mllbb.root")
f.keys()

In [9]:
f['tt_sm'].values()

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 1.65619962e-01, 3.98957677e-01, 2.18590814e+00,
       1.58196862e+01, 6.05252197e+01, 1.47047312e+02, 2.89332146e+02,
       4.25423718e+02, 5.36738716e+02, 5.82233750e+02, 5.96902241e+02,
       5.78876190e+02, 5.33198175e+02, 4.95775481e+02, 4.37184665e+02,
       4.06098907e+02, 3.55080124e+02, 2.99513143e+02, 2.64388152e+02,
       2.34903520e+02, 2.08291048e+02, 1.73219161e+02, 1.49875313e+02,
       1.29935738e+02, 1.11951377e+02, 9.94483519e+01, 8.44970230e+01,
       6.89218223e+01, 6.58453133e+01, 5.35424746e+01, 4.64990042e+01,
       4.06876308e+01, 3.64100179e+01, 2.89093209e+01, 2.49918112e+01,
       2.35047453e+01, 1.86409015e+01, 1.47667620e+01, 1.36968664e+01,
       1.22038457e+01, 1.13281681e+01, 9.69850076e+00, 7.19901311e+00,
       8.40215203e+00, 6.02053900e+00, 5.06297708e+01])

In [11]:
for k in f.keys():
    if "tt" in k:
        print(f"{k}: {f[k].values().sum()}")

tt_sm;1: 7760.908303241831
tt_sm_elecIDUp;1: 8035.724930260858
tt_sm_elecIDDown;1: 7492.995587147609
tt_sm_muonIDUp;1: 7760.908303241831
tt_sm_muonIDDown;1: 7760.908303241831
tt_sm_muonISOUp;1: 7760.908303241831
tt_sm_muonISODown;1: 7760.908303241831
tt_sm_trigSFUp;1: 7819.726536569966
tt_sm_trigSFDown;1: 7702.330917764175
tt_sm_L1prefireUp;1: 7709.281092561425
tt_sm_L1prefireDown;1: 7812.197207312064
tt_sm_PUUp;1: 7745.058064374889
tt_sm_PUDown;1: 7777.078356317723
tt_sm_jetPuIDUp;1: 7768.242610348505
tt_sm_jetPuIDDown;1: 7753.580065639401
tt_sm_btagSFbc_correlatedUp;1: 7851.171738281466
tt_sm_btagSFbc_correlatedDown;1: 7671.291622189544
tt_sm_btagSFlight_correlatedUp;1: 7767.93727167933
tt_sm_btagSFlight_correlatedDown;1: 7753.846227556907
tt_sm_btagSFbc_2017Up;1: 7877.379765991531
tt_sm_btagSFbc_2017Down;1: 7645.245541289471
tt_sm_btagSFlight_2017Up;1: 7765.638039586497
tt_sm_btagSFlight_2017Down;1: 7756.164686675951
tt_sm_FSRUp;1: 7916.334685293403
tt_sm_FSRDown;1: 7519.55972813440

In [30]:
CATSELECTED = ['ee_2j_2b_mllbb']
CATSELECTED = sorted(CATSELECTED)

In [34]:
with open(os.path.join('/users/hnelson2/ttbarEFT-coffea2025/datacard_maker/test_cards/','TESTING-ttbar-ee_2j_2b_mllbb.json'), 'r') as file:
        scalings_content = json.load(file)

# new_scalings = []
# for ch_index, channel_name in enumerate(CATSELECTED, start=1):
#     # find all items that match this channel
#     matches = [item for item in scalings_content if item.get("channel") == channel_name]

#     if not matches:
#         raise ValueError(f"Channel '{channel_name}' not found in scalings_content")

#     # update and append in order
#     for item in matches:
#         new_item = item.copy()
#         new_item["channel"] = f"ch{ch_index}"
#         new_scalings.append(new_item)

# scalings_content = new_scalings

# print(scalings_content)

# with open(os.path.join(ptzlj0pt_path, 'scalings.json'), 'w') as file:
#     json.dump(scalings_content, file, indent=4)

In [46]:
scalings_content[0]['scaling'][5]

[1.0,
 5.32725287409039e-10,
 6.004943996609364e-10,
 -7.167073281853848e-10,
 5.102661596492579e-10,
 -2.9375755651848496e-10,
 2.587500440045472e-10,
 3.601164444691869e-10,
 1.8007780868322116e-10,
 3.78175150103941e-10,
 1.2813193539603454e-09,
 3.87694164137011e-10,
 -4.835502437210533e-10,
 -7.470271506610892e-10,
 1.3543376344444709e-09,
 6.287902891117768e-11,
 -3.31507171840303e-10,
 1.112614743113618e-09,
 -5.525728886851245e-10,
 -6.172081691565874e-10,
 -6.547619199921282e-10,
 -1.1541641294692131e-10,
 2.2737255330294612e-10,
 -3.8602278718744597e-10,
 8.55562191323473e-10,
 -3.221970799259286e-10,
 1.0236661510789503e-09,
 -2.6692412189851397e-10,
 -0.21012562391340925,
 1.8328737473168288e-09,
 9.135641945263493e-10,
 2.0608077788137693e-09,
 4.95850533459259e-10,
 -1.7486128453203944e-10,
 7.379912690275029e-10,
 0.04614505636251472,
 0.04126941842857944,
 1.002891457901153e-10,
 -2.8473473251731713e-10,
 -1.3473387434681669e-09,
 2.0089167475827404e-11,
 5.128254556032

In [5]:
pkl_file = "../analysis/SR_2017_2j2b.pkl.gz"
prepared_pkl = flatten_vineReduce_hists(pkl_file)

In [6]:
hists = pickle.load(gzip.open(prepared_pkl))

In [7]:
hists.keys()

dict_keys(['njets', 'nleps', 'nbjets', 'mll', 'mllZ', 'mllbb', 'ptll', 'l0pt', 'l0eta', 'l0phi', 'l1pt', 'l1eta', 'l1phi', 'j0pt', 'j0eta', 'j0phi', 'mt2', 'MET'])

In [8]:
hists = hists['mllbb']

In [9]:
hists
wc_list = hists[{'channel':'ee_2j_2b','process':'TT01j2lmtt0to700_UL17','systematic':'nominal'}]._wc_names.keys()

In [16]:
scalings = hists[{'channel':'ee_2j_2b','process':'TT01j2lmtt0to700_UL17','systematic':'nominal'}].make_scaling(flow='show')

In [24]:
hists[{'channel':'ee_2j_2b','process':'TT01j2lmtt0to700_UL17','systematic':'nominal'}].values(flow=True)[:,1:-1]

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 1.37494036e+00, -4.63065936e-10,  1.18313927e-10, ...,
        -5.16966162e-04,  1.01283811e-09,  8.55938123e-04],
       [ 2.46615131e-01,  1.36829604e-03,  4.52097679e-03, ...,
         5.61718227e-11,  3.06931851e-10, -2.58280711e-10],
       [ 7.35160165e+00,  3.24529818e-03,  8.49964292e-03, ...,
        -5.47948388e-03,  7.11567665e-10,  1.32269860e-02]],
      shape=(52, 153))

In [25]:
scalings

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 1.00000000e+00, -1.68394917e-10,  8.60502251e-11, ...,
        -1.87995850e-04,  3.68320745e-10,  6.22527455e-04],
       [ 1.00000000e+00,  2.77415265e-03,  1.83321144e-02, ...,
         1.13885597e-10,  6.22289170e-10, -1.04730277e-09],
       [ 1.00000000e+00,  2.20720486e-04,  1.15616206e-03, ...,
        -3.72672796e-04,  4.83954177e-11,  1.79919786e-03]],
      shape=(52, 153))

In [27]:
hists = pickle.load(gzip.open(prepared_pkl))
hists = hists['njets']
hists[{'channel':'ee_2j_2b','process':'TT01j2lmtt0to700_UL17','systematic':'nominal'}].values(flow=True)[:,1:-1]

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 7.48875361e+01, -1.35049394e-02,  2.06661664e-01, ...,
         1.01053214e-02, -7.82254258e-09,  6.74431551e-02],
       [ 1.47447228e+01,  5.73171629e-02,  7.08235349e-02, ...,
         1.77788060e-03,  2.65033965e-09,  2.07731993e-02],
       [ 5.30985667e+00,  1.56320495e-09,  1.61117947e-09, ...,
         2.69235809e-05, -1.84118658e-09,  5.93350788e-04]],
      shape=(10, 153))

In [48]:
pkl_file = "../analysis/SR_2017_2j2b.pkl.gz"
# var = 'mllbb'
var = 'njets'
# ch = 'ee_2j_2b'
out_dir = './test_multi_cards/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

prepared_pkl = flatten_vineReduce_hists(pkl_file)

obj = DatacardMaker(
    prepared_pkl,
    year_lst=[],         # Processes all years found in pkl
    do_nuisance=True,    # Set to False if you want to skip systematics entirely for now
    do_mc_stat = True,
    out_dir=out_dir
)

Opening: /users/hnelson2/ttbarEFT-coffea2025/ttbarEFT/params/rate_systs.json
Opening: flat_hists.pkl.gz
Pkl Open Time: 0.92 s
Loading: njets
Skipping (ignored): dataUL17
Correlating years
('ttUL17', 'ee_2j_2b', 'btagSFbc_2017Up') -- 
('ttUL17', 'ee_2j_2b', 'btagSFbc_2017Down') -- 
('ttUL17', 'ee_2j_2b', 'btagSFlight_2017Up') -- 
('ttUL17', 'ee_2j_2b', 'btagSFlight_2017Down') -- 
('ttUL17', 'mm_2j_2b', 'btagSFbc_2017Up') -- 
('ttUL17', 'mm_2j_2b', 'btagSFbc_2017Down') -- 
('ttUL17', 'mm_2j_2b', 'btagSFlight_2017Up') -- 
('ttUL17', 'mm_2j_2b', 'btagSFlight_2017Down') -- 
('ttUL17', 'em_2j_2b', 'btagSFbc_2017Up') -- 
('ttUL17', 'em_2j_2b', 'btagSFbc_2017Down') -- 
('ttUL17', 'em_2j_2b', 'btagSFlight_2017Up') -- 
('ttUL17', 'em_2j_2b', 'btagSFlight_2017Down') -- 
('tWUL17', 'ee_2j_2b', 'btagSFbc_2017Up') -- 
('tWUL17', 'ee_2j_2b', 'btagSFbc_2017Down') -- 
('tWUL17', 'ee_2j_2b', 'btagSFlight_2017Up') -- 
('tWUL17', 'ee_2j_2b', 'btagSFlight_2017Down') -- 
('tWUL17', 'mm_2j_2b', 'btagSFbc_201

In [52]:
var_lst = ['njets']
dists = var_lst if len(var_lst) else dc.hists.keys()

In [57]:
ch_lst = []
selected_wcs = {}
for km_dist in dists:
    all_chs = obj.channels(km_dist)
    matched_chs = regex_match(all_chs,ch_lst)
    
    dist_wcs = obj.get_selected_wcs(km_dist,matched_chs)
    for p,wcs in dist_wcs.items():
        if not p in selected_wcs:
            selected_wcs[p] = []
        for wc in wcs:
            if not wc in selected_wcs[p]:
                selected_wcs[p].append(wc)
                
with open(os.path.join(out_dir,"selectedWCs.txt"),"w") as f:
    selected_wcs_for_json = {}
    for p,v in selected_wcs.items():
        if not obj.is_signal(p):
            # WC selection will include backgrounds in the dict (always with an empty list), so remove them here
            continue
        selected_wcs_for_json[p] = list(v)
    json.dump(selected_wcs_for_json,f)

Selecting WCs from subset of channels: ['ee_2j_2b', 'mm_2j_2b', 'em_2j_2b']
WC Selection Time: 0.01 s


In [60]:
wcs_dict

{'tt': ['ctGRe', 'ctu8'],
 'tW': ['ctGRe', 'ctu8'],
 'DY': ['ctGRe', 'ctu8'],
 'Diboson': ['ctGRe', 'ctu8'],
 'Triboson': ['ctGRe', 'ctu8'],
 'TTll': ['ctGRe', 'ctu8'],
 'WWZ': ['ctGRe', 'ctu8']}

In [67]:
def run_local(dc,km_dists,channels,selected_wcs, crop_negative_bins, wcs_dict):
    for km_dist in km_dists:
        all_chs = dc.channels(km_dist)
        matched_chs = regex_match(all_chs,channels)
        if channels:
            print(f"Channels to process: {matched_chs}")
        for ch in matched_chs:
            r = dc.analyze(km_dist,ch,selected_wcs, crop_negative_bins, wcs_dict)

# wcs_dict is used to fix the WC to a specific set of values when calculating the yields - generally leave this empty
run_local(obj,dists,matched_chs,selected_wcs, crop_negative_bins=True, wcs_dict={})

print("Making scalings-preselect.json file...")
with open(os.path.join(out_dir,"scalings-preselect.json"),"w") as f:
    json.dump(obj.scalings, f, indent=4)

Channels to process: ['ee_2j_2b', 'mm_2j_2b', 'em_2j_2b']
Analyzing njets in ee_2j_2b
ch_hist:HistEFT(
  StrCategory(['tt', 'tW', 'DY', 'Diboson', 'Triboson', 'TTll', 'WWZ'], growth=True, name='process'),
  StrCategory(['ee_2j_2b'], growth=True, name='channel'),
  StrCategory(['nominal', 'sumw2', 'elecIDUp', 'elecIDDown', 'muonIDUp', 'muonIDDown', 'muonISOUp', 'muonISODown', 'trigSFUp', 'trigSFDown', 'L1prefireUp', 'L1prefireDown', 'PUUp', 'PUDown', 'jetPuIDUp', 'jetPuIDDown', 'btagSFbc_correlatedUp', 'btagSFbc_correlatedDown', 'btagSFlight_correlatedUp', 'btagSFlight_correlatedDown', 'btagSFbc_2017Up', 'btagSFbc_2017Down', 'btagSFlight_2017Up', 'btagSFlight_2017Down', 'FSRUp', 'FSRDown', 'ISRUp', 'ISRDown', 'renormUp', 'renormDown', 'factUp', 'factDown', 'hdampUp', 'hdampDown'], growth=True, name='systematic', label='Systematic Uncertainty'),
  Regular(8, 0, 8, name='njets'),
  Integer(0, 153, name='quadratic_term'),
  storage=Double()) # Sum: 238.0
Generating root file: TESTING-ttbar

In [65]:
# my_wcs = ["ctGRe", "ctu8"] 
# wcs_dict = {p: my_wcs for p in obj.processes(var)}
# # print(wcs_dict)

# print(f"Generating card for channel: {ch}")
# obj.analyze(
#     km_dist=var,
#     ch=ch,
#     selected_wcs=wcs_dict,
#     crop_negative_bins=True,
#     wcs_dict=wcs_dict
# )

In [70]:
CATSELECTED = ['ee_2j_2b_njets', 'em_2j_2b_njets', 'mm_2j_2b_njets']
CATSELECTED = sorted(CATSELECTED)

with open(os.path.join('/users/hnelson2/ttbarEFT-coffea2025/datacard_maker/test_multi_cards/','scalings-preselect.json'), 'r') as file:
        scalings_content = json.load(file)

new_scalings = []
for ch_index, channel_name in enumerate(CATSELECTED, start=1):
    # find all items that match this channel
    matches = [item for item in scalings_content if item.get("channel") == channel_name]

    if not matches:
        raise ValueError(f"Channel '{channel_name}' not found in scalings_content")

    # update and append in order
    for item in matches:
        new_item = item.copy()
        new_item["channel"] = f"ch{ch_index}"
        new_scalings.append(new_item)

scalings_content = new_scalings

print(scalings_content)

with open(os.path.join('/users/hnelson2/ttbarEFT-coffea2025/datacard_maker/test_multi_cards/', 'scalings.json'), 'w') as file:
    json.dump(scalings_content, file, indent=4)

[{'channel': 'ch1', 'process': 'tt_sm', 'parameters': ['cSM[1]', 'cQd1[0,-10.0,10.0]', 'ctj1[0,-10.0,10.0]', 'cQj31[0,-10.0,10.0]', 'ctj8[0,-10.0,10.0]', 'ctd1[0,-10.0,10.0]', 'ctd8[0,-10.0,10.0]', 'ctGRe[0,-10.0,10.0]', 'ctGIm[0,-10.0,10.0]', 'cQj11[0,-10.0,10.0]', 'cQj18[0,-10.0,10.0]', 'ctu8[0,-10.0,10.0]', 'cQd8[0,-10.0,10.0]', 'ctu1[0,-10.0,10.0]', 'cQu1[0,-10.0,10.0]', 'cQj38[0,-10.0,10.0]', 'cQu8[0,-10.0,10.0]'], 'scaling': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0